In [ ]:
from mg_sampling import PROJECT_DIR
import seaborn as sns
import pandas as pd
from ast import literal_eval
import itertools

In [ ]:
systems = ['7ez0', '7ez0_noion', '7ez0_2', '7ez0_noion_2']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p', 'ecc_spce']
repids = [0]

df = []
for system, forcefield, repid in itertools.product(systems, forcefields, repids):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{repid:02d}'
    _df = pd.read_csv(path / 'ion_analysis.csv', na_filter=False, converters={'inner_names': literal_eval, 'inner_resids': literal_eval, 'inner_resnames': literal_eval})
    _df['system'] = system.removesuffix('_2')
    _df['forcefield'] = forcefield
    _df['repid'] = repid
    _df['replicate'] = 1 if system.endswith('_2') else 0
    df.append(_df)
df = pd.concat(df)
for col in ['name', 'mgrna', 'forcefield', 'system']:
    df[col] = df[col].astype('category')


In [ ]:
df.head()

In [ ]:
tstart = 100000

grouped = df.query(f"time >= {tstart}").groupby(['name', 'n_inner', 'forcefield', 'system', 'replicate']).size()
grouped = grouped.reset_index(name='counts')

g = sns.FacetGrid(grouped, row='name', col='system')
sns.set_style("ticks", {'axes.grid': True})

def plot_fn(data, **kws):
    kws['color'] = None
    sns.barplot(data.query("replicate == 0"), fill=True, alpha=0.8, **kws)
    sns.barplot(data.query("replicate == 1"), fill=False, linewidth=1, **kws)


g.map_dataframe(
    plot_fn,
    x='n_inner',
    y='counts',
    hue='forcefield',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

In [ ]:
tstart = 100000

grouped = df.query(f"time >= {tstart}")[['name', 'inner_names', 'forcefield', 'system', 'replicate']]

grouped = grouped.explode('inner_names').groupby(['name', 'inner_names', 'forcefield', 'system', 'replicate']).size()


grouped = grouped.reset_index(name='counts')

In [ ]:
g = sns.FacetGrid(grouped, row='name', col='system', aspect=3, row_order=['MG', 'NA'])
sns.set_style("ticks", {'axes.grid': True})

def plot_fn(data, **kws):
    kws['color'] = None
    sns.barplot(data.query("replicate == 0"), fill=True, alpha=0.8, **kws)
    sns.barplot(data.query("replicate == 1"), fill=False, linewidth=1, **kws)


g.map_dataframe(
    plot_fn,
    x='inner_names',
    y='counts',
    hue='forcefield',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"Site (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

In [ ]:

grouped = df.query("name == 'MG' & n_inner > 0 & time > 100000")[['name', 'forcefield', 'system', 'replicate', 'mgrna']]
grouped = grouped.groupby(['name', 'forcefield', 'system', 'replicate', 'mgrna']).size().reset_index(name='counts')
grouped['mgrna'] = grouped['mgrna'].astype(str)
grouped.loc[grouped['counts'] < 5, 'mgrna'] = 'other'

g = sns.FacetGrid(grouped, row='name', col='system', aspect=3, row_order=['MG', 'NA'])
sns.set_style("ticks", {'axes.grid': True})

def plot_fn(data, **kws):
    kws['color'] = None
    sns.barplot(data.query("replicate == 0"), fill=True, alpha=0.8, **kws)
    sns.barplot(data.query("replicate == 1"), fill=False, linewidth=1, **kws)


g.map_dataframe(
    plot_fn,
    x='mgrna',
    y='counts',
    hue='forcefield',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"MgRNA classification")
g.set_titles("{row_name} ({col_name})")
for ax in g.axes.flatten():
    ax.tick_params(labelbottom=True)

In [ ]:
grouped = df.groupby(['time', 'name', 'system', 'replicate', 'forcefield']).size().reset_index(name='counts')
grouped.set_index('time')

g = sns.FacetGrid(grouped, row='name', col='system')
sns.set_style("ticks", {'axes.grid': True})

def plot_fn(data, **kws):
    kws['color'] = None
    sns.lineplot(data.query("replicate == 0"), alpha=0.5, **kws)
    sns.lineplot(data.query("replicate == 1"), alpha=0.5, **kws)


g.map_dataframe(
    plot_fn,
    x='time',
    y='counts',
    hue='forcefield',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

Already at the beginning there are inner-sphere bound MG… Thanks to genion??

In [ ]:
df.query('name == "MG" & time == 0 & n_inner > 0 & system == "7ez0_noion"')